# Модуль 08 · Урок 3. Итоговый проект 3. Проверяемый ETL-конвейер

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## Данные для самостоятельного исследования

В архиве есть отдельный увеличенный CSV-набор этого проекта. Сначала пройдите задания на автопроверку, затем примените написанные функции к этим данным, проверьте агрегаты и объясните полученные результаты. Это не дополнительное оценочное задание: обязательные пять заданий проверяются автоматически ниже.

In [ ]:
project_raw = pd.read_csv(ROOT/'data'/'capstone_dirty.csv',keep_default_na=True)
print('Сырых записей:',len(project_raw))
print(project_raw.head(6))

## Сценарий проекта
Компания присылает неаккуратную таблицу товаров: лишние пробелы в артикуле, разные регистры категорий, текстовые цены, пустые количества, повторные записи. Нужно построить **воспроизводимую обработку**, которая отдельно фиксирует плохие записи и формирует итоговую витрину. Нельзя выдавать пропуски за ноль и нельзя стирать исходные значения до аудита.

## 1. Договорённость о схеме
Вход: `item`, `price`, `quantity`, `category`, `email`. Выход: `item` — очищенный артикул в верхнем регистре; `category` — очищенный нижний регистр, `unknown` для NA; `price_num` и `qty_num` — числа; `line_total` — произведение. Записи с отсутствующим `item`, ошибочной или отрицательной ценой, ошибочным или отрицательным количеством отправляются в карантин. Для дублей по нормализованному item выбирается **последняя** валидная запись.

## 2. Разделение стадий
Реализуйте очистку текста, преобразование чисел, проверку валидности, дедупликацию и составление сводного отчёта в отдельных функциях. Это облегчает поиск ошибок и независимое тестирование. Каждую функцию можно запускать на входе с новым индексом: не полагайтесь на последовательную нумерацию строк, если задание не просит её сбросить.

In [ ]:
import pandas as pd
sample = pd.DataFrame({'code':[' x ',' Y '], 'price':['12.5','bad']})
clean = sample.assign(code=lambda x:x.code.str.strip().str.upper(),price_num=lambda x:pd.to_numeric(x.price,errors='coerce'))
print(clean)

## 3. Не допускайте незаметной потери строк
Карантин — не «удалённые без объяснения» записи. Для каждой стадии укажите число входных строк, число невалидных строк и количество дублей. В больших системах это превращается в журнал качества и метрики мониторинга. Отчёт должен показывать как число строк, так и число уникальных объектов после дедупликации.

## 4. Проверка повторного запуска
Очистка строк должна быть идемпотентной. Дедупликация после однократного выполнения не должна удалять новые строки при втором запуске. Порядок операций также важен: сперва нормализуйте item, затем ищите дубли — иначе `X-8` и ` x-8 ` будут выглядеть разными.

**Критерии сдачи:** правильные пять шагов, неизменность входа, одинаковый результат на двух различных наборах данных, ясность обработки ошибок.

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m8_11 · normalize inventory

Этап 1. В новом DataFrame нормализуйте item (strip+upper) и category (strip+lower, NA→unknown). Все исходные колонки оставьте.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m8_11: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m8_11 = expected_for("m8_11", df=df)


In [ ]:
# m8_11: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 5 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`.

```text
    item   price quantity category          email
11   A-1  128.75        2     food  A@EXAMPLE.COM
2    B-2                3     food        b@x.ru 
3    A-1  120.50        2     food  A@EXAMPLE.COM
8    C-3  ошибка     None      toy            bad
10  None       0        7  unknown               
15   D-4   -4.00        1      toy           None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m8_11", result, _expected_m8_11)

### m8_12 · type inventory

Этап 2. Добавьте `price_num` и `qty_num`, используя `pd.to_numeric(...,errors="coerce")`; исходные текстовые колонки сохраните.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m8_12: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m8_12 = expected_for("m8_12", df=df)


In [ ]:
# m8_12: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 7 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`, `price_num`, `qty_num`.

```text
     item   price quantity category          email  price_num  qty_num
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM     128.75      2.0
2     b-2                3     food        b@x.ru         NaN      3.0
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM     120.50      2.0
8     C-3  ошибка     None      TOY            bad        NaN      NaN
10   None       0        7     None                      0.00      7.0
15   d-4    -4.00        1      toy           None      -4.00      1.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m8_12", result, _expected_m8_12)

### m8_13 · valid inventory

Этап 3. Отфильтруйте валидные товары: item не пуст после strip, цена и quantity после to_numeric числовые и >=0. Получите только валидные исходные строки.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m8_13: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m8_13 = expected_for("m8_13", df=df)


In [ ]:
# m8_13: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 5 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`.

```text
     item   price quantity category          email
11   A-1   128.75        2    FOOD   A@EXAMPLE.COM
3    A-1   120.50        2    FOOD   A@EXAMPLE.COM
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m8_13", result, _expected_m8_13)

### m8_14 · latest products

Этап 4. Нормализуйте item через strip+upper, удалите повторы по очищенному item, оставив последнюю запись, затем сбросьте индекс. Оставьте все остальные поля.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m8_14: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m8_14 = expected_for("m8_14", df=df)


In [ ]:
# m8_14: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 5 строк, 5 столбцов; колонки: `item`, `price`, `quantity`, `category`, `email`.

```text
   item   price quantity category          email
0   B-2                3     food        b@x.ru 
1   A-1  120.50        2    FOOD   A@EXAMPLE.COM
2   C-3  ошибка     None      TOY            bad
3  None       0        7     None               
4   D-4   -4.00        1      toy           None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m8_14", result, _expected_m8_14)

### m8_15 · inventory summary

Этап 5. Постройте dict `source_rows` (все строки), `valid_rows` (item непуст, числовые price и quantity >=0), `unique_valid_items` (разных нормализованных item среди валидных).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `item`, `price`, `quantity`, `category`, `email`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m8_15: используются переменные df

df = pd.DataFrame({
    'item': [' A-1 ', 'b-2', ' A-1 ', 'C-3', None, ' d-4 '],
    'price': ['128.75', '', '120.50', 'ошибка', '0', '-4.00'],
    'quantity': ['2', '3', '2', None, '7', '1'],
    'category': [' FOOD ', 'food', ' FOOD ', 'TOY', None, 'toy'],
    'email': ['A@EXAMPLE.COM', ' b@x.ru ', 'A@EXAMPLE.COM', 'bad', '', None],
}, index=[11, 2, 3, 8, 10, 15])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m8_15 = expected_for("m8_15", df=df)


In [ ]:
# m8_15: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

```python
{'source_rows': 6, 'valid_rows': 2, 'unique_valid_items': 1}
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m8_15", result, _expected_m8_15)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()